# ISL Sign Recognition — Improved Training for 81%+ Accuracy (263 classes)

**Target model:** `include_no_cnn_transformer_small.pth`

**Improvements over original:**
- AdamW optimizer with weight decay
- Warmup + CosineAnnealing LR scheduler
- Higher augmentation probability (0.5 vs 0.4)
- Gradient clipping
- More epochs (75) with higher patience (15)
- Mixup augmentation on tensors

**Branch:** `cnn_new` | **Time:** ~60 min on Colab GPU

In [ ]:
# ── Step 1: Clone cnn_new branch ─────────────────────────────────────
!git clone --branch cnn_new --single-branch https://github.com/BishalDubey27/Major_Project.git
%cd Major_Project
!ls

In [ ]:
# ── Step 2: Install dependencies ─────────────────────────────────────
!pip install mediapipe==0.10.31 transformers==4.44.0 timm joblib tqdm scikit-learn -q
print('Done')

In [ ]:
# ── Step 3: Load keypoints from Google Drive (3 separate folders) ─────
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

# ── Change these paths to match your Drive folder locations ──────────
DRIVE_TRAIN = '/content/drive/MyDrive/include_train_keypoints'
DRIVE_VAL   = '/content/drive/MyDrive/include_val_keypoints'
DRIVE_TEST  = '/content/drive/MyDrive/include_test_keypoints'

os.makedirs('/content/keypoint', exist_ok=True)
shutil.copytree(DRIVE_TRAIN, '/content/keypoint/include_train_keypoints')
shutil.copytree(DRIVE_VAL,   '/content/keypoint/include_val_keypoints')
shutil.copytree(DRIVE_TEST,  '/content/keypoint/include_test_keypoints')

for split in ['include_train_keypoints', 'include_val_keypoints', 'include_test_keypoints']:
    n = len([f for f in os.listdir(f'/content/keypoint/{split}') if f.endswith('.json')])
    print(f'{split}: {n} files')

In [ ]:
# ── Step 4: Improved training pipeline ───────────────────────────────
import os, sys, json, math
import torch
import torch.nn.functional as F
from torch.utils import data as torch_data
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import numpy as np

sys.path.insert(0, '/content/Major_Project/INCLUDE')
sys.path.insert(0, '/content/Major_Project')

from models.transformer import Transformer
from configs import TransformerConfig
from dataset import KeypointsDataset
from utils import seed_everything, AverageMeter, EarlyStopping, load_json
from augment import Augmentation, OneOf, plus7rotation, minus7rotation, gaussSample, cutout, upsample, downsample

os.chdir('/content/Major_Project/INCLUDE')

DATASET       = 'include'
DATA_DIR      = '/content/keypoint'
SAVE_PATH     = '/content'
EPOCHS        = 75
BATCH_SIZE    = 128
LR            = 1e-4
WEIGHT_DECAY  = 1e-2
WARMUP_EPOCHS = 5
PATIENCE      = 15
GRAD_CLIP     = 1.0
MIXUP_ALPHA   = 0.2
SEED          = 42
SIZE          = 'small'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
seed_everything(SEED)

label_map = load_json(f'label_maps/label_map_{DATASET}.json')
n_classes  = len(label_map)
idx_to_label = {v: k for k, v in label_map.items()}
print(f'Classes: {n_classes}')

config = TransformerConfig(size=SIZE)
model  = Transformer(config=config, n_classes=n_classes).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

class ImprovedKeypointsDataset(KeypointsDataset):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.augs = [
            Augmentation(OneOf(plus7rotation, minus7rotation), p=0.5),
            Augmentation(gaussSample, p=0.5),
            Augmentation(cutout, p=0.5),
            Augmentation(OneOf(upsample, downsample), p=0.5),
        ]

train_ds = ImprovedKeypointsDataset(os.path.join(DATA_DIR, f'{DATASET}_train_keypoints'), use_augs=True,  label_map=label_map, mode='train')
val_ds   = KeypointsDataset(         os.path.join(DATA_DIR, f'{DATASET}_val_keypoints'),   use_augs=False, label_map=label_map, mode='val')
train_loader = torch_data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = torch_data.DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(EPOCHS - WARMUP_EPOCHS, 1)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def mixup_batch(x, y, alpha=0.2):
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

save_file      = os.path.join(SAVE_PATH, f'include_no_cnn_transformer_{SIZE}_improved.pth')
early_stopping = EarlyStopping(patience=PATIENCE, mode='max')
best_val_acc   = 0

# Track history for plots
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}

for epoch in range(EPOCHS):
    model.train()
    train_losses = AverageMeter(); train_accs = AverageMeter()
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]')
    for batch in pbar:
        x, y = batch['data'].to(device), batch['label'].to(device)
        x, y_a, y_b, lam = mixup_batch(x, y, MIXUP_ALPHA)
        optimizer.zero_grad()
        preds = model(x)
        loss = lam * F.cross_entropy(preds, y_a, label_smoothing=0.1) + \
               (1 - lam) * F.cross_entropy(preds, y_b, label_smoothing=0.1)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        train_losses.update(loss.item())
        train_accs.update(accuracy_score(y.cpu().numpy(), preds.detach().cpu().argmax(-1).numpy()))
        pbar.set_postfix(loss=f'{train_losses.avg:.4f}', acc=f'{train_accs.avg:.4f}')
    scheduler.step()

    model.eval()
    val_losses = AverageMeter(); val_accs = AverageMeter()
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Val]'):
            x, y = batch['data'].to(device), batch['label'].to(device)
            preds = model(x)
            val_losses.update(F.cross_entropy(preds, y).item())
            val_accs.update(accuracy_score(y.cpu().numpy(), preds.cpu().argmax(-1).numpy()))

    history['train_loss'].append(train_losses.avg)
    history['val_loss'].append(val_losses.avg)
    history['train_acc'].append(train_accs.avg)
    history['val_acc'].append(val_accs.avg)
    history['lr'].append(scheduler.get_last_lr()[0])

    print(f'Epoch {epoch+1}: train_loss={train_losses.avg:.4f} train_acc={train_accs.avg:.4f} | val_loss={val_losses.avg:.4f} val_acc={val_accs.avg:.4f}')

    if val_accs.avg > best_val_acc:
        best_val_acc = val_accs.avg
        torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(), 'score': best_val_acc}, save_file)
        print(f'  Saved best: {best_val_acc:.4f}')

    early_stopping(save_file, val_accs.avg, model, optimizer, scheduler)
    if early_stopping.early_stop:
        print('Early stopping triggered'); break

print(f'Best val accuracy: {round(best_val_acc*100,2)}%')

In [ ]:
# ── Step 5: Training curves ───────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

epochs_ran = range(1, len(history['train_acc']) + 1)

fig = plt.figure(figsize=(18, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig)

# Accuracy curve
ax1 = fig.add_subplot(gs[0])
ax1.plot(epochs_ran, history['train_acc'], label='Train Acc', color='steelblue')
ax1.plot(epochs_ran, history['val_acc'],   label='Val Acc',   color='orange')
ax1.axhline(best_val_acc, color='green', linestyle='--', label=f'Best Val {best_val_acc:.3f}')
ax1.set_title('Accuracy'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Loss curve
ax2 = fig.add_subplot(gs[1])
ax2.plot(epochs_ran, history['train_loss'], label='Train Loss', color='steelblue')
ax2.plot(epochs_ran, history['val_loss'],   label='Val Loss',   color='orange')
ax2.set_title('Loss'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(True, alpha=0.3)

# LR curve
ax3 = fig.add_subplot(gs[2])
ax3.plot(epochs_ran, history['lr'], color='purple')
ax3.set_title('Learning Rate'); ax3.set_xlabel('Epoch'); ax3.set_ylabel('LR')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

In [ ]:
# ── Step 6: Full evaluation — accuracy, F1, confusion matrix, pie chart
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score
)
import seaborn as sns

cp = torch.load(save_file, map_location='cpu', weights_only=False)
model.load_state_dict(cp['model'])
model.eval()

test_ds = KeypointsDataset(
    os.path.join(DATA_DIR, f'{DATASET}_test_keypoints'),
    use_augs=False, label_map=label_map, mode='test'
)
test_loader = torch_data.DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

all_preds = []; all_labels = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Evaluating'):
        x, y = batch['data'].to(device), batch['label']
        preds = model(x).cpu().argmax(-1)
        all_preds.extend(preds.numpy())
        all_labels.extend(y.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

test_acc  = (all_preds == all_labels).mean()
f1_macro  = f1_score(all_labels, all_preds, average='macro',  zero_division=0)
f1_weighted = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
recall    = recall_score(all_labels, all_preds, average='weighted', zero_division=0)

print('=' * 50)
print(f'Test Accuracy  : {test_acc*100:.2f}%')
print(f'F1 (macro)     : {f1_macro:.4f}')
print(f'F1 (weighted)  : {f1_weighted:.4f}')
print(f'Precision (w)  : {precision:.4f}')
print(f'Recall (w)     : {recall:.4f}')
print('=' * 50)

In [ ]:
# ── Step 7: Pie chart — correct vs incorrect ──────────────────────────
correct   = int((all_preds == all_labels).sum())
incorrect = len(all_labels) - correct

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart
axes[0].pie(
    [correct, incorrect],
    labels=[f'Correct\n{correct} ({test_acc*100:.1f}%)', f'Incorrect\n{incorrect} ({(1-test_acc)*100:.1f}%)'],
    colors=['#2ecc71', '#e74c3c'],
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[0].set_title('Test Set: Correct vs Incorrect Predictions', fontsize=13, fontweight='bold')

# Bar chart — metrics summary
metrics = ['Accuracy', 'F1 Macro', 'F1 Weighted', 'Precision', 'Recall']
values  = [test_acc, f1_macro, f1_weighted, precision, recall]
colors  = ['#3498db', '#9b59b6', '#e67e22', '#1abc9c', '#e74c3c']
bars = axes[1].bar(metrics, values, color=colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontweight='bold')
axes[1].set_ylim(0, 1.1)
axes[1].set_title('Evaluation Metrics', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Score')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/content/metrics_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Step 8: Confusion matrix (top 30 classes by frequency) ───────────
# Show top 30 most frequent classes to keep it readable
from collections import Counter

top_n = 30
top_class_ids = [c for c, _ in Counter(all_labels.tolist()).most_common(top_n)]
top_class_names = [idx_to_label[i] for i in top_class_ids]

mask = np.isin(all_labels, top_class_ids)
filtered_labels = all_labels[mask]
filtered_preds  = all_preds[mask]

# Remap to 0..top_n-1
remap = {old: new for new, old in enumerate(top_class_ids)}
filtered_labels = np.array([remap[l] for l in filtered_labels])
filtered_preds  = np.array([remap[p] if p in remap else -1 for p in filtered_preds])
valid = filtered_preds >= 0
filtered_labels = filtered_labels[valid]
filtered_preds  = filtered_preds[valid]

cm = confusion_matrix(filtered_labels, filtered_preds, labels=list(range(top_n)))
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)

fig, ax = plt.subplots(figsize=(20, 16))
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=top_class_names, yticklabels=top_class_names,
    linewidths=0.5, ax=ax, annot_kws={'size': 7}
)
ax.set_title(f'Confusion Matrix — Top {top_n} Classes (Normalized)', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Step 9: Per-class F1 score bar chart (top 20 and bottom 20) ───────
report = classification_report(all_labels, all_preds, output_dict=True, zero_division=0)

class_f1 = {}
for k, v in report.items():
    if k.isdigit():
        class_f1[idx_to_label[int(k)]] = v['f1-score']

sorted_f1 = sorted(class_f1.items(), key=lambda x: x[1], reverse=True)
top20    = sorted_f1[:20]
bottom20 = sorted_f1[-20:]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, data, title, color in [
    (axes[0], top20,    'Top 20 Classes by F1 Score',    '#2ecc71'),
    (axes[1], bottom20, 'Bottom 20 Classes by F1 Score', '#e74c3c')
]:
    labels_plot = [d[0] for d in data]
    values_plot = [d[1] for d in data]
    bars = ax.barh(labels_plot, values_plot, color=color, edgecolor='white')
    for bar, val in zip(bars, values_plot):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.2f}', va='center', fontsize=8)
    ax.set_xlim(0, 1.15)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('F1 Score')
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('/content/per_class_f1.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Step 10: Download model and all plots ─────────────────────────────
from google.colab import files

for f in [save_file, '/content/training_curves.png', '/content/metrics_summary.png',
          '/content/confusion_matrix.png', '/content/per_class_f1.png']:
    files.download(f)
    print('Downloaded:', f)

print()
print('Next steps:')
print('1. Place .pth in Major_Project/INCLUDE/')
print('2. Rename to: include_no_cnn_transformer_small.pth')
print('3. Sign recognition will work at improved accuracy!')